# Biden Tweet Sentiment Analysis

This notebook analyzes approximately 498K tweets mentioning Joe Biden using three complementary sentiment/emotion classification methods:
- **NRCLex**: Lexicon-based emotion detection
- **TextBlob**: Polarity-based sentiment classification
- **RoBERTa**: Transformer-based emotion classification using a Twitter-optimized model

In [ ]:
# ── Setup & Dependencies ──────────────────────────────────────────
!pip install -q nrclex langdetect langid swifter transformers torch

# Standard libraries
import pandas as pd
import re
import numpy as np
from collections import Counter

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('punkt')
nltk.download('stopwords')

# Sentiment & Emotion Analysis
from nrclex import NRCLex
from textblob import TextBlob
from transformers import pipeline

# Language Detection
import langid
import swifter

print("✓ All dependencies loaded successfully")

In [ ]:
# ── Configuration ──────────────────────────────────────────
DATA_PATH = "/content/drive/MyDrive/Research Paper (AI + Crypto)/Copy of lessprocessedjoe.csv"

# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

# Load dataset
data_biden = pd.read_csv(DATA_PATH, lineterminator="\n")
print(f"Dataset shape: {data_biden.shape}")
data_biden.head()

In [ ]:
# ── Language Filtering ──────────────────────────────────────────
# Filter to English tweets only
data_biden['lang'] = data_biden['tweet'].astype(str).swifter.apply(lambda x: langid.classify(x)[0])
data_biden = data_biden[data_biden['lang'] == 'en'].reset_index(drop=True)
print(f"English tweets: {data_biden.shape[0]:,}")

## 1. NRCLex Emotion Classification

NRCLex is a lexicon-based approach that identifies the dominant emotion in text using the National Research Council's Emotion Lexicon. It classifies emotions into categories such as anger, anticipation, disgust, fear, joy, negative, positive, sadness, surprise, and trust.

In [ ]:
def nrc_dominant(text):
    """Return the dominant NRC emotion for a given text."""
    if not isinstance(text, str) or not text.strip():
        return np.nan
    emo = NRCLex(text)
    if emo.raw_emotion_scores:
        return max(emo.raw_emotion_scores, key=emo.raw_emotion_scores.get)
    return np.nan

data_biden['nrc_emotion'] = data_biden['tweet'].astype(str).apply(nrc_dominant)
print(data_biden['nrc_emotion'].value_counts())

In [ ]:
plt.figure(figsize=(10, 6))
freq = data_biden['nrc_emotion'].value_counts().sort_index()
freq.plot(kind='bar', color='skyblue', edgecolor='black')
plt.title("NRCLex Emotion Distribution — Biden Tweets")
plt.xlabel("Emotion")
plt.ylabel("Frequency")
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## 2. TextBlob Sentiment Analysis

TextBlob provides a simple polarity-based sentiment analysis approach. Tweets are classified as Positive (polarity > 0), Negative (polarity < 0), or Neutral (polarity = 0), where polarity ranges from -1 to 1.

In [ ]:
def textblob_sentiment(text):
    """Classify text as Positive, Negative, or Neutral using TextBlob polarity."""
    polarity = TextBlob(text).sentiment.polarity
    if polarity > 0:
        return "Positive"
    elif polarity < 0:
        return "Negative"
    return "Neutral"

data_biden['textblob_sentiment'] = data_biden['tweet'].astype(str).apply(textblob_sentiment)
print(data_biden['textblob_sentiment'].value_counts())

In [ ]:
counts = data_biden['textblob_sentiment'].value_counts()
plt.figure(figsize=(8, 5))
plt.bar(counts.index, counts.values, color=['#2ecc71', '#e74c3c', '#95a5a6'])
plt.title("TextBlob Sentiment Distribution — Biden Tweets")
plt.xlabel("Sentiment")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

## 3. RoBERTa Transformer Classification

We use our fine-tuned RoBERTa model (`roberta-tweet-emotion-finetuned`) for emotion classification. This model was fine-tuned on 100K political tweets from our dataset (see `04_roberta_finetuning.ipynb`) starting from `cardiffnlp/twitter-roberta-base`, a RoBERTa variant pre-trained on ~58M tweets.

Fine-tuning on domain-specific political tweets allows the model to capture sentiment patterns unique to political discourse that generic models may miss.

In [ ]:
# Load fine-tuned RoBERTa model (trained in 04_roberta_finetuning.ipynb)
FINETUNED_MODEL_PATH = "/content/drive/MyDrive/Research Paper (AI + Crypto)/roberta-tweet-emotion-finetuned"

classifier = pipeline(
    "text-classification",
    model=FINETUNED_MODEL_PATH,
    tokenizer=FINETUNED_MODEL_PATH,
    top_k=None
)

def roberta_dominant(text):
    """Return the dominant emotion predicted by the fine-tuned RoBERTa model."""
    if not isinstance(text, str) or not text.strip():
        return np.nan
    preds = classifier(text[:512])
    return max(preds[0], key=lambda d: d['score'])['label']

data_biden['roberta_emotion'] = data_biden['tweet'].apply(roberta_dominant)
print(data_biden['roberta_emotion'].value_counts())

In [ ]:
plt.figure(figsize=(10, 6))
freq = data_biden['roberta_emotion'].value_counts()
freq.plot(kind='bar', color='coral', edgecolor='black')
plt.title("RoBERTa Emotion Distribution — Biden Tweets")
plt.xlabel("Emotion")
plt.ylabel("Number of Tweets")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Summary

This analysis compared three distinct approaches to sentiment and emotion classification:
- **NRCLex** provides lexicon-based, fine-grained emotion categories
- **TextBlob** offers simple polarity-based sentiment (positive/negative/neutral)
- **RoBERTa** leverages deep learning for nuanced, context-aware emotion detection

The resulting classifications are stored in `data_biden['nrc_emotion']`, `data_biden['textblob_sentiment']`, and `data_biden['roberta_emotion']` respectively, enabling comparative analysis of how different methods perceive the sentiment landscape of Biden-related tweets.